# 09 — Explainability (SHAP)
**Objective:** Generate SHAP explanations for all three disease models.

In [ ]:
import shap, joblib, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

MDL = Path('../ml/models')
for disease in ['ckd','diabetes','hypertension']:
    mpath = MDL / f'{disease}_model.joblib'
    ppath = MDL / f'{disease}_preprocessor.joblib'
    if not mpath.exists():
        print(f'⚠️  {disease} model not found. Run train_models.py first.')
        continue
    model = joblib.load(mpath)
    prep  = joblib.load(ppath)
    print(f'✅ {disease.upper()} model loaded')

## SHAP TreeExplainer — Global Feature Importance

In [ ]:
# Example using CKD model (run for each disease)
disease = 'ckd'
model = joblib.load(MDL / f'{disease}_model.joblib')
prep  = joblib.load(MDL / f'{disease}_preprocessor.joblib')

# Create a small synthetic sample for demonstration
import pandas as pd
sample = pd.DataFrame([{
    'age':55,'gender_enc':1,'bmi':28,'systolic_bp':140,'diastolic_bp':90,
    'glucose':120,'creatinine':1.8,'hemoglobin':11,'potassium':4.5,
    'sodium':138,'urea':40,'albumin':3.5
}])

# Transform sample
try:
    X_sample = prep.transform(sample)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    sv_arr = sv[1][0] if isinstance(sv, list) else sv[0]
    for feat, val in sorted(zip(sample.columns, sv_arr),
                            key=lambda x: abs(x[1]), reverse=True):
        direction = '↑ RISK' if val > 0 else '↓ RISK'
        print(f"  {feat:25s}: SHAP = {val:+.4f}  {direction}")
except Exception as e:
    print(f'Note: {e}')
    print('SHAP requires the base model (not calibrated wrapper) for TreeExplainer.')
    print('See ml/inference.py for the implementation used in production.')

## SHAP Explanation Note
SHAP values indicate model feature contributions — NOT causal medical mechanisms.

Always include this disclaimer when presenting SHAP results:

> *'The model identified this variable as an important contributor to the estimated risk. This does not mean the variable caused the condition.'*